# EDA — Default of Credit Card Clients

**Source:** UCI *Default of Credit Card Clients* — 30,000 credit card holders in Taiwan, billing
months April–September 2005.

**One row = one client.** The target is whether that client defaulted on their *next* month's
payment (October 2005).

**Why the original load failed:** `data/credit_card_clients.xls` is a real legacy Excel workbook
(OLE2), not a CSV with an `.xls` name. `pd.read_csv` tried to decode it as UTF-8 text and choked on
the first byte (`0xd0`, the Excel file signature). It needs `pd.read_excel`, which for `.xls` relies
on the `xlrd` engine (now added to `pyproject.toml`).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

## 1. Load the data

Two quirks of this file:

- it is an Excel workbook, so `read_excel` (not `read_csv`), reading the sheet named `Data`
- row 0 holds placeholder names (`X1`, `X2`, …) and the *real* header is row 1 → `header=1`

The target column ships as `default payment next month` (with spaces). Renaming it to
`DEFAULT_NEXT_MONTH` makes it easier to reuse in the training script later.

In [ ]:
DATA_PATH = Path("../data/credit_card_clients.xls")

dataset = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
dataset = dataset.rename(columns={"default payment next month": "DEFAULT_NEXT_MONTH"})

dataset.shape

In [ ]:
dataset.head()

## 2. What the columns mean

| column | meaning |
| --- | --- |
| `ID` | row identifier — not a feature, drop it before training |
| `LIMIT_BAL` | credit limit in NT dollars (covers the client plus family/supplementary cards) |
| `SEX` | 1 = male, 2 = female |
| `EDUCATION` | 1 = graduate school, 2 = university, 3 = high school, 4 = other, 0/5/6 = undocumented |
| `MARRIAGE` | 1 = married, 2 = single, 3 = other, 0 = undocumented |
| `AGE` | age in years |
| `PAY_0`, `PAY_2` … `PAY_6` | repayment status for Sep, Aug, Jul, Jun, May, Apr 2005 |
| `BILL_AMT1` … `BILL_AMT6` | bill statement amount (NT$) for Sep … Apr |
| `PAY_AMT1` … `PAY_AMT6` | amount actually paid (NT$) for Sep … Apr |
| `DEFAULT_NEXT_MONTH` | **target**: 1 = defaulted next month, 0 = paid |

Two things that trip people up:

- **The `PAY_*` columns count backwards in time.** `PAY_0` / `BILL_AMT1` / `PAY_AMT1` are the most
  recent month (September), index 6 is the oldest (April). And there is no `PAY_1` — the September
  column is called `PAY_0`. That naming is in the original dataset, not a mistake here.
- **The `PAY_*` codes:** `-1` = paid in full, `1`–`8` = that many months late. The values `-2` and
  `0` appear in the data but are not in the official documentation; they are generally read as
  "no credit used that month" and "paid the minimum / revolving credit".

In [ ]:
dataset.info()

## 3. Data quality

Range checks matter more than a missing-value count here — the file has no nulls, but it does hold
category codes that the documentation never defines, and negative bill amounts (overpayments and
refunds).

In [ ]:
quality = pd.DataFrame({
    "dtype": dataset.dtypes.astype(str),
    "missing": dataset.isna().sum(),
    "unique": dataset.nunique(),
    "min": dataset.min(numeric_only=True),
    "max": dataset.max(numeric_only=True),
})
quality

In [ ]:
print("rows, columns  :", dataset.shape)
print("missing cells  :", int(dataset.isna().sum().sum()))
print("duplicate rows :", int(dataset.duplicated().sum()))
print("duplicate IDs  :", int(dataset["ID"].duplicated().sum()))

## 4. The target: how many clients default?

This is the single most important number for modelling — it is imbalanced, roughly one default in
five. A model that predicts "never defaults" would already score ~78% accuracy, which is why
accuracy is the wrong metric for this project (use ROC-AUC, precision/recall instead).

In [ ]:
counts = dataset["DEFAULT_NEXT_MONTH"].value_counts().sort_index()
share = dataset["DEFAULT_NEXT_MONTH"].value_counts(normalize=True).sort_index()

pd.DataFrame({"clients": counts, "share": share.round(4)}).rename(
    index={0: "0 - paid", 1: "1 - defaulted"}
)

## 5. Who is in the data?

The coded categories are unreadable as raw integers, so map them to labels first. `labelled` is a
copy used only for inspection — the numeric `dataset` stays untouched for modelling.

In [ ]:
SEX_LABELS = {1: "male", 2: "female"}
EDUCATION_LABELS = {
    1: "graduate school", 2: "university", 3: "high school", 4: "other",
    0: "undocumented", 5: "undocumented", 6: "undocumented",
}
MARRIAGE_LABELS = {1: "married", 2: "single", 3: "other", 0: "undocumented"}

labelled = dataset.assign(
    SEX_LABEL=dataset["SEX"].map(SEX_LABELS),
    EDUCATION_LABEL=dataset["EDUCATION"].map(EDUCATION_LABELS),
    MARRIAGE_LABEL=dataset["MARRIAGE"].map(MARRIAGE_LABELS),
)

for column in ["SEX_LABEL", "EDUCATION_LABEL", "MARRIAGE_LABEL"]:
    print(labelled[column].value_counts(dropna=False), end="\n\n")

## 6. Default rate by group

Counts alone say little; what matters is how the default rate moves between groups. Read the
`clients` column alongside it — a 26% default rate over 323 clients is far shakier evidence than the
same rate over 14,000.

In [ ]:
def default_rate_by(column, frame=labelled):
    summary = frame.groupby(column, observed=True)["DEFAULT_NEXT_MONTH"].agg(
        clients="size", default_rate="mean"
    )
    return summary.assign(default_rate=summary["default_rate"].round(3)).sort_values(
        "clients", ascending=False
    )


for column in ["SEX_LABEL", "EDUCATION_LABEL", "MARRIAGE_LABEL"]:
    print(default_rate_by(column), end="\n\n")

In [ ]:
age_bands = pd.cut(dataset["AGE"], bins=[20, 30, 40, 50, 60, 100], right=False)

dataset.groupby(age_bands, observed=True)["DEFAULT_NEXT_MONTH"].agg(
    clients="size", default_rate="mean"
).round(3)

## 7. Repayment history — the strongest signal

`PAY_*` is where the predictive power sits. Most clients cluster in `-2`/`-1`/`0` (no credit used,
paid in full, or revolving), and the delay codes thin out fast.

In [ ]:
PAY_COLUMNS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

pay_status = pd.DataFrame(
    {column: dataset[column].value_counts() for column in PAY_COLUMNS}
).sort_index().fillna(0).astype(int)
pay_status

In [ ]:
# PAY_0 is last month's repayment status - the most recent, most informative column.
dataset.groupby("PAY_0")["DEFAULT_NEXT_MONTH"].agg(
    clients="size", default_rate="mean"
).round(3)

## 8. The money columns

Bills and payments are heavily right-skewed: the median bill is around NT$22,000 while the maximum
is over NT$1.6 million. Negative `BILL_AMT` values are overpayments or refunds, not errors. Skew
like this is worth remembering when choosing a model — trees handle it, linear models usually want
a log or scaling step.

In [ ]:
MONEY_COLUMNS = (
    ["LIMIT_BAL"]
    + [f"BILL_AMT{i}" for i in range(1, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 7)]
)

dataset[["AGE"] + MONEY_COLUMNS].describe().T.round(0)

## 9. What correlates with defaulting?

Linear correlation only catches straight-line relationships, so treat this as a rough ranking rather
than proof. It still makes the picture clear.

In [ ]:
correlations = (
    dataset.drop(columns=["ID"])
    .corr(numeric_only=True)["DEFAULT_NEXT_MONTH"]
    .drop("DEFAULT_NEXT_MONTH")
)

correlations.sort_values(key=abs, ascending=False).round(3).to_frame("corr_with_default")

## 10. Takeaways

1. **30,000 rows, 24 usable features, no missing values and no duplicates.** The data is clean; the
   work is in encoding and framing, not repair.
2. **The target is imbalanced (~22% default).** Score with ROC-AUC and precision/recall, not
   accuracy, and consider `class_weight="balanced"`.
3. **Recent repayment status dominates.** `PAY_0` correlates at ~0.33 with defaulting and the
   default rate climbs from ~13% (paid/no credit) to ~69% at two months late. The `PAY_*` columns
   fade in importance the further back they go.
4. **Credit limit is the strongest inverse signal** (~-0.15): higher limits default less, which
   mostly reflects the bank's own prior assessment of the client.
5. **Demographics move the needle only slightly** — a few percentage points between groups, and the
   extreme rates sit on tiny undocumented buckets. Don't over-read them.
6. **Bill and payment amounts barely correlate on their own.** Ratios (payment ÷ bill, bill ÷ limit,
   utilisation trend across the six months) are likely to carry more than the raw amounts.

**Next:** drop `ID`, treat `SEX` / `EDUCATION` / `MARRIAGE` as categorical rather than numeric,
split the data with stratification on the target, and move on to step 1 of `STEPS.md` — training a
model and logging the runs to MLflow.